# 🧹 Limpieza y preparación de datos con Python

Este notebook documenta el proceso de limpieza, revisión y preparación realizado con Python sobre los datos del Centro de Recuperación de Fauna Silvestre de Bizkaia correspondientes al periodo 2022–2025.

El objetivo es mostrar las principales comprobaciones y transformaciones realizadas antes de continuar con el análisis en Power BI.

Los archivos de datos no se incluyen en este repositorio.

## 📥 1. Carga y unificación de los datos

Los datos fueron trabajados inicialmente de forma independiente para cada año. En esta fase se cargaron los archivos correspondientes a 2022, 2023, 2024 y 2025 y se unificaron en un único DataFrame para realizar las comprobaciones sobre el periodo completo.

> **Nota:** Los archivos de datos no se incluyen en este repositorio. Este notebook tiene carácter documental y muestra el proceso de preparación realizado durante el análisis original. Por este motivo, las celdas que requieren los archivos de datos no pueden ejecutarse directamente desde GitHub.

In [ ]:
import pandas as pd

# Carga de los datasets anuales
df2022 = pd.read_csv("fauna_2022_clean.csv")
df2023 = pd.read_csv("fauna_2023_clean.csv")
df2024 = pd.read_csv("fauna_2024_clean.csv")
df2025 = pd.read_csv("fauna_2025_clean.csv")

# Unificación de los cuatro años
df = pd.concat(
    [df2022, df2023, df2024, df2025],
    ignore_index=True
)

## 🔎 2. Comprobación inicial de la estructura

Una vez unificados los datos, se revisó la estructura del DataFrame y sus dimensiones para comprobar que la carga y concatenación de los cuatro archivos se habían realizado correctamente.

In [ ]:
df.info()

In [ ]:
df.shape

**Resultado:** El conjunto unificado contiene 6.276 registros y 21 columnas.

### Corrección de la codificación de caracteres

Durante la preparación inicial de los datos se detectaron problemas en la representación de algunos caracteres especiales, especialmente en nombres de especies y otros campos de texto. Algunos caracteres como `ñ` y determinadas letras acentuadas aparecían representados mediante símbolos incorrectos.

Para conservar correctamente estos caracteres, el DataFrame unificado se exportó utilizando la codificación `UTF-8-SIG`.

In [ ]:
df.to_csv(
    "fauna_bizkaia_2022_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

## 🧩 3. Comprobación de valores faltantes

Se revisaron los valores ausentes de todas las columnas para identificar posibles problemas de calidad y determinar qué campos requerían una revisión más detallada.

La presencia de valores faltantes no implica necesariamente un error, por lo que se analizaron especialmente aquellas columnas en las que podían afectar a la interpretación posterior de los datos.

In [ ]:
df.isna().sum()

### Resultado

La comprobación identificó valores faltantes en varias variables. Los principales casos fueron:

| Variable | Valores faltantes |
|---|---:|
| ESPECIE | 12 |
| NOMBRE CIENTIFICO | 107 |
| CAUSA INGRESO | 378 |
| CODIGO MUNICIPIO RECOGIDA | 17 |
| MUNICIPIO RECOGIDA | 18 |
| EVOLUCION | 2.831 |
| CODIGO PROVINCIA SUELTA | 4.859 |
| PROVINCIA SUELTA | 4.859 |
| CODIGO MUNICIPIO SUELTA | 4.869 |
| MUNICIPIO SUELTA | 4.869 |

No todos los valores faltantes se consideraron un problema de calidad. Su ausencia se revisó teniendo en cuenta la naturaleza de cada variable y su relevancia para el análisis posterior.

### Revisión de nombres científicos ausentes

Se detectaron 107 registros sin información en `NOMBRE CIENTIFICO`. En lugar de asumir que estos valores constituían un error, se revisó qué especies estaban asociadas a dichos registros para identificar posibles patrones.

In [ ]:
df.loc[df["NOMBRE CIENTIFICO"].isna(), "ESPECIE"].value_counts()

**Resultado:** La mayor parte de los registros sin nombre científico correspondía a `Trachemis Scripta (Híbrida)` (49 registros) y `Murciélago (Gro. Pipistrelus)` (41 registros).

## 🔤 4. Normalización de categorías

Durante la revisión de los valores únicos se detectaron diferencias de formato en algunas variables categóricas. Una misma categoría podía aparecer escrita con mayúsculas o minúsculas diferentes, lo que podía generar categorías duplicadas durante el análisis.

Se normalizaron estas variables para asegurar una clasificación homogénea.

### Revisión de valores únicos en `CLASE`

Se revisaron los valores de la variable `CLASE` para identificar categorías aparentemente iguales pero representadas con diferente formato.

In [ ]:
df["CLASE"].value_counts()

**Resultado:** Se detectaron diferencias de capitalización en categorías como `Aves`/`AVES`, `Mamiferos`/`MAMIFEROS` y `Reptiles`/`REPTILES`.

Estas diferencias podían provocar que una misma categoría se contabilizara como si fueran valores distintos.

### Normalización de `CLASE`

In [ ]:
df["CLASE"] = df["CLASE"].str.title()

In [ ]:
df["CLASE"].value_counts()

**Resultado:** La normalización permitió agrupar las categorías que únicamente se diferenciaban por el uso de mayúsculas y minúsculas.

### Normalización de `SUBCLASE`

Se aplicó la misma revisión a la variable `SUBCLASE`, donde también se detectaron categorías duplicadas debido a diferencias de capitalización.

In [ ]:
df["SUBCLASE"].value_counts()

In [ ]:
df["SUBCLASE"] = df["SUBCLASE"].str.title()

In [ ]:
df["SUBCLASE"].value_counts()

**Resultado:** La normalización permitió unificar las categorías que presentaban diferencias de capitalización.

## 📊 5. Revisión de variables relevantes

Una vez normalizadas las categorías, se revisaron algunas variables relacionadas directamente con el análisis posterior para comprobar su distribución y detectar posibles inconsistencias de formato.

Se prestó especial atención a `CAUSA INGRESO`, `EVOLUCION` y `ESTADO INGRESO`, ya que estas variables permiten analizar las circunstancias de los ingresos y su evolución posterior.

### Causa de ingreso

Se revisó la distribución de `CAUSA INGRESO` para conocer las principales causas registradas y comprobar la consistencia de sus categorías.

In [ ]:
df["CAUSA INGRESO"].value_counts()

**Resultado:** Se identificaron 31 categorías de causa de ingreso. Las más frecuentes fueron:

- Cría: 1.680 registros
- Traumatismo: 1.039 registros
- Control poblacional: 543 registros
- Captura: 512 registros
- Debilidad: 374 registros
- Atropello: 300 registros

Estas variables se conservaron para el análisis posterior de las principales causas de ingreso.

### Evolución de los animales ingresados

Se revisaron los valores de `EVOLUCION` para identificar posibles diferencias de formato entre categorías equivalentes.

In [ ]:
df["EVOLUCION"].value_counts()

**Hallazgo:** Se detectó una diferencia de capitalización en la categoría `No entra en el Centro-Intervención Guardería`, que aparecía representada en dos formatos.

In [ ]:
df["EVOLUCION"] = df["EVOLUCION"].str.title()

In [ ]:
df["EVOLUCION"].value_counts()

**Resultado:** Tras normalizar la capitalización, las dos variantes quedaron agrupadas bajo una única categoría: `No Entra En El Centro-Intervención Guardería`, con 96 registros.

### Estado de ingreso

Finalmente, se revisó la distribución de `ESTADO INGRESO` para comprobar los valores disponibles en esta variable.

In [ ]:
df["ESTADO INGRESO"].value_counts()

**Resultado:**

| Estado | Registros |
|---|---:|
| VIVO | 3.447 |
| MUERTO | 2.829 |

## 🆔 6. Revisión de identificadores

Durante la comprobación de calidad se revisó si el identificador `ID` aparecía repetido en el conjunto de datos unificado.

La presencia de identificadores repetidos no se consideró automáticamente un problema, ya que los datos corresponden a diferentes años y era necesario comprobar si el identificador podía repetirse entre periodos.

### Comprobación de registros duplicados

Antes de analizar la unicidad del identificador, se comprobó si existían filas completamente duplicadas en el conjunto de datos.

In [ ]:
df.duplicated().sum()

**Resultado:** No se detectaron registros completamente duplicados.

In [ ]:
df["ID"].duplicated().sum()

**Resultado:** Se detectaron 4.587 apariciones de valores de `ID` que ya habían aparecido previamente en el conjunto de datos unificado.

### Creación de un identificador único

El identificador `ID` se reinicia al comienzo de cada año, por lo que un mismo número puede aparecer en diferentes periodos.

Por este motivo, utilizar únicamente `ID` para identificar los registros del periodo 2022–2025 podía hacer que un mismo número fuese interpretado como un registro repetido.

Para garantizar un identificador único para cada ingreso, se creó una nueva variable combinando el año con el `ID` original.

In [ ]:
df["ID_UNICO"] = (
    df["AÑO"].astype(str)
    + "-"
    + df["ID"].astype(str)
)

**Ejemplo:** un registro con `ID` 25 en 2022 y otro con `ID` 25 en 2023 pasan a identificarse como `2022-25` y `2023-25`, respectivamente.

## ✅ 7. Dataset preparado para el análisis

Tras completar las comprobaciones y transformaciones anteriores, se obtuvo el conjunto de datos preparado para continuar con el análisis mediante Power BI.

El dataset final incorpora las correcciones de formato realizadas durante la preparación y el identificador `ID_UNICO`, creado para distinguir de forma inequívoca los ingresos de los diferentes años.

El archivo de datos final no se incluye en este repositorio.